# 10 · Token tagger: which parts of a text did an AI write?

A fusion text is a human text with one or more AI-written sections (GhostWriter, Section 3.2). Notebook 08 recovered
those sections for the training fusion texts by comparing them word by word with their human source. Here XLM-RoBERTa
base is fine-tuned as a *token tagger*: instead of one label per text it predicts, for every token, the probability
that it was written by an AI (as in SemEval-2024 Task 8, subtask C). Training targets: every token of a human text is
human, every token of an ai text is AI, and in a fusion text the tokens inside the recovered sections are AI.

For each validation and test text the token probabilities are summarized into a few numbers (share of AI tokens,
longest AI run, highest and lowest smoothed AI probability, ...). A human text should have almost no AI tokens, an ai
text almost only AI tokens, and a fusion text one or more AI sections. These numbers become features of the stacker.
Long texts are read in 512-token chunks, as in notebooks 03 and 06.

In [ ]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-generated-text-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwkit import env
from hwkit.download import ensure_data
from hwkit.tracker import Run
P = env.setup(seed=42)
ensure_data(P)

In [ ]:
# Imports, config; training sample (fusion texts with recovered AI sections, human and ai texts), validation, test
os.environ["TOKENIZERS_PARALLELISM"] = "false"     # the tokenizer runs before DataLoader workers are forked
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import logging as hub_logging

from hwkit.config import load_config
from hwkit.split import LABELS, load_split
from hwkit.spans import parse_spans

# library info messages (the token classification layer is new and untrained, download hints); errors still show
hf_logging.set_verbosity_error()
hub_logging.set_verbosity_error()
assert torch.cuda.is_available(), "turn on the GPU (Kaggle: Settings -> Accelerator -> GPU T4 or P100)"
DEV = "cuda"
NOTEBOOK_START = time.time()
cfg = load_config("tagger_xlmr")
TC, DC = cfg["train"], cfg["data"]

train = load_split(pd.read_csv(P.data / "train.csv"))
span_of = pd.read_csv(P.repo / "results" / "fusion_spans.csv.gz").set_index("id")["spans"]   # NaN: no AI section found
part = train[train["fold"] != 0]
fit = pd.concat([part[(part["label"] == "fusion") & part["id"].isin(span_of.dropna().index)].sample(n=DC["n_fusion"], random_state=42),
                 part[part["label"] == "human"].sample(n=DC["n_human"], random_state=42),
                 part[part["label"] == "ai"].sample(n=DC["n_ai"], random_state=42)]).reset_index(drop=True)
val = train[train["fold"] == 0].reset_index(drop=True)
test = pd.read_csv(P.data / "test.csv")
del train, part
monitor_idx = np.random.default_rng(42).choice(len(val), TC["monitor_size"], replace=False)
run = Run(cfg["name"], cfg, P)
print(f"training: {fit['label'].value_counts().to_dict()}; validation: {len(val)}; test: {len(test)}; "
      f"GPU: {torch.cuda.get_device_name()}")

## Token labels

In [ ]:
# Tokenize every text without length limit; a token is AI if its first character lies inside an AI section
tok = AutoTokenizer.from_pretrained(cfg["model"]["arch"])
BODY = cfg["model"]["max_len"] - 2            # room left after the start and end tokens
IGNORE = -100                                 # label of special and padding tokens, skipped by the loss

def token_labels(offsets, label, spans):
    """0 = human, 1 = AI for every token; None for a fusion text without recovered sections."""
    if label == "human":
        return np.zeros(len(offsets), dtype=np.int8)
    if label == "ai":
        return np.ones(len(offsets), dtype=np.int8)
    if not isinstance(spans, str):
        return None
    starts = np.array([o[0] for o in offsets])
    lab = np.zeros(len(offsets), dtype=np.int8)
    for s, e in parse_spans(spans):
        lab[(starts >= s) & (starts < e)] = 1
    return lab

def tokenize(df, batch=2000):
    """Token ids (int32 arrays) and token labels (int8 arrays, or None when unknown or for test)."""
    ids, labs = [], []
    has_label = "label" in df
    for i in range(0, len(df), batch):
        part = df.iloc[i:i + batch]
        enc = tok(part["text"].tolist(), add_special_tokens=False, return_offsets_mapping=True, verbose=False)
        for k, (x, off) in enumerate(zip(enc["input_ids"], enc["offset_mapping"])):
            ids.append(np.asarray(x, dtype=np.int32))
            labs.append(token_labels(off, part["label"].iat[k], span_of.get(part["id"].iat[k])) if has_label else None)
    return ids, labs

def chunk_starts(n):
    """Start positions of the chunks covering a text of n tokens; the last chunk ends exactly at the text's end."""
    if n <= BODY:
        return [0]
    return list(range(0, n - BODY, BODY)) + [n - BODY]

def wrap(x):
    """Add the start and end tokens to a chunk."""
    return np.concatenate([[tok.cls_token_id], x, [tok.sep_token_id]]).astype(np.int64)

def wrap_labels(lab):
    return np.concatenate([[IGNORE], lab, [IGNORE]]).astype(np.int64)

t0 = time.time()
data = {name: tokenize(df) for name, df in (("fit", fit), ("val", val), ("test", test))}
for df in (fit, val, test):
    df.drop(columns="text", inplace=True)              # token ids and labels are all that is needed from here on
print(f"tokenized in {(time.time() - t0) / 60:.1f} min")
lab_fit = pd.Series([l.mean() for l in data["fit"][1]], index=fit.index)
lab_fit.groupby(fit["label"]).describe()[["mean", "25%", "50%", "75%"]].reindex(LABELS).round(3).rename_axis(
    "share of AI tokens")

In [ ]:
# Datasets: one random chunk per training text and epoch; every chunk of every text for prediction
class RandomChunk(Dataset):
    def __init__(self, token_lists, label_lists):
        self.x, self.y = token_lists, label_lists
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        x = self.x[i]
        start = int(torch.randint(0, max(len(x) - BODY, 0) + 1, (1,)))    # torch RNG: differs per worker
        return wrap(x[start:start + BODY]), wrap_labels(self.y[i][start:start + BODY])

def collate(batch):
    """Pad to the longest chunk of the batch; padding gets the IGNORE label and attention mask 0."""
    xs, ys = zip(*batch)
    width = max(len(x) for x in xs)
    input_ids = torch.full((len(xs), width), tok.pad_token_id, dtype=torch.long)
    labels = torch.full((len(xs), width), IGNORE, dtype=torch.long)
    mask = torch.zeros((len(xs), width), dtype=torch.long)
    for i, (x, y) in enumerate(zip(xs, ys)):
        input_ids[i, :len(x)] = torch.from_numpy(x)
        labels[i, :len(y)] = torch.from_numpy(y)
        mask[i, :len(x)] = 1
    return input_ids, mask, labels

@torch.no_grad()
def token_probs(token_lists):
    """AI probability of every token of every text (later chunks overwrite the overlap of the last chunk)."""
    chunks = [(d, s, x[s:s + BODY]) for d, x in enumerate(token_lists) for s in chunk_starts(len(x))]
    order = np.argsort([len(c[2]) for c in chunks], kind="stable")   # similar lengths together: less padding
    loader = DataLoader([(wrap(chunks[i][2]), np.zeros(0, dtype=np.int64)) for i in order],
                        batch_size=TC["eval_batch_size"], collate_fn=collate, num_workers=2)
    out = [np.zeros(len(x), dtype=np.float32) for x in token_lists]
    model.eval()
    pos = 0
    for input_ids, mask, _ in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=input_ids.to(DEV), attention_mask=mask.to(DEV)).logits
        p = torch.softmax(logits.float(), dim=-1)[..., 1].cpu().numpy()
        for row in range(len(p)):
            d, s, c = chunks[order[pos]]
            out[d][s:s + len(c)] = p[row, 1:1 + len(c)]
            pos += 1
    return out

def token_scores(probs, labels):
    """Token-level precision, recall and F1 of the AI class over all texts with known labels."""
    y = np.concatenate([l for l in labels if l is not None])
    p = np.concatenate([pr for pr, l in zip(probs, labels) if l is not None]) > 0.5
    tp, fp, fn = (p & (y == 1)).sum(), (p & (y == 0)).sum(), (~p & (y == 1)).sum()
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"token_precision": prec, "token_recall": rec, "token_f1": 2 * prec * rec / max(prec + rec, 1e-9)}

## Fine-tuning

The same recipe as the document classifier (notebook 03): AdamW, learning rate 2e-5 with warmup and linear decay,
mixed precision, one random 512-token chunk per text and epoch, a checkpoint after every epoch. The loss is the
cross-entropy over all real tokens. After each epoch the token-level F1 of the AI class is measured on a fixed sample
of validation texts (learning curve only).

In [ ]:
# Model, optimizer, schedule; resume from the last epoch checkpoint if there is one
model = AutoModelForTokenClassification.from_pretrained(
    cfg["model"]["arch"], num_labels=2, id2label={0: "human", 1: "ai"}, label2id={"human": 0, "ai": 1}).to(DEV)
loader = DataLoader(RandomChunk(*data["fit"]), batch_size=TC["batch_size"], shuffle=True, drop_last=True,
                    collate_fn=collate, num_workers=2, pin_memory=True, generator=torch.Generator().manual_seed(42))
steps = len(loader) * TC["epochs"]
no_decay = ("bias", "LayerNorm.weight")                    # usual practice: no weight decay on these
params = [{"params": [p for n, p in model.named_parameters() if not n.endswith(no_decay)],
           "weight_decay": TC["weight_decay"]},
          {"params": [p for n, p in model.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0}]
opt = torch.optim.AdamW(params, lr=TC["lr"])
sched = get_linear_schedule_with_warmup(opt, int(TC["warmup"] * steps), steps)
scaler = torch.amp.GradScaler("cuda")
start_epoch = 0
if not run.is_done("train") and run.ckpt("last.pt").exists():
    state = torch.load(run.ckpt("last.pt"), map_location="cpu", weights_only=False)
    for obj, key in ((model, "model"), (opt, "opt"), (sched, "sched"), (scaler, "scaler")):
        obj.load_state_dict(state[key])
    start_epoch = state["epoch"]
    del state
print(f"{len(loader)} steps per epoch, {TC['epochs']} epochs; starting at epoch {start_epoch + 1}")

In [ ]:
# Training loop; token-level F1 of the AI class on the monitor sample after each epoch
if not run.is_done("train"):
    mon_ids = [data["val"][0][i] for i in monitor_idx]
    mon_lab = [data["val"][1][i] for i in monitor_idx]
    for epoch in range(start_epoch, TC["epochs"]):
        model.train()
        t0, loss_sum, seen = time.time(), 0.0, 0
        for input_ids, mask, labels in loader:
            input_ids, mask, labels = input_ids.to(DEV), mask.to(DEV), labels.to(DEV)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(input_ids=input_ids, attention_mask=mask).logits
            loss = F.cross_entropy(logits.float().flatten(0, 1), labels.flatten(), ignore_index=IGNORE)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scale = scaler.get_scale()
            scaler.step(opt)                      # skipped when float16 gradients overflowed
            scaler.update()
            if scaler.get_scale() >= scale:       # move the schedule only when the optimizer really stepped
                sched.step()
            loss_sum += loss.item() * len(input_ids)
            seen += len(input_ids)
        s = token_scores(token_probs(mon_ids), mon_lab)
        run.log_epoch(epoch + 1, {"train_loss": loss_sum / seen, **{f"monitor_{k}": v for k, v in s.items()},
                                  "minutes": (time.time() - t0) / 60})
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": epoch + 1}, run.ckpt("last.pt"))
        print(f"epoch {epoch + 1}: loss {loss_sum / seen:.4f}, monitor token F1 {s['token_f1']:.4f}, "
              f"{(time.time() - t0) / 60:.1f} min")
    torch.save({k: v.half() for k, v in model.state_dict().items()}, run.ckpt("model.pt"))   # weights only
    run.ckpt("last.pt").unlink()                  # optimizer state is no longer needed (saves 2 GB of output)
    run.mark_done("train")
else:
    model.load_state_dict({k: v.float() for k, v in
                           torch.load(run.ckpt("model.pt"), map_location="cpu", weights_only=True).items()})
run.history.round(4)

## Token probabilities and text features for validation and test

In [ ]:
# AI probability of every token of every validation and test text, summarized per text
def text_features(p, window=cfg["features"]["smooth_window"]):
    hard = np.concatenate([[False], p > 0.5, [False]]).astype(np.int8)
    edges = np.flatnonzero(np.diff(hard))                      # starts and ends of runs of AI tokens
    runs = edges[1::2] - edges[0::2]
    smooth = np.convolve(p, np.ones(window) / window, mode="valid") if len(p) >= window else np.array([p.mean()])
    longest = runs.max() if len(runs) else 0
    return {"tag_mean": p.mean(), "tag_share": (p > 0.5).mean(), "tag_std": p.std(), "tag_runs": len(runs),
            "tag_longest_share": longest / len(p), "tag_longest_log": np.log1p(longest),
            "tag_smooth_max": smooth.max(), "tag_smooth_min": smooth.min()}

feats, probs = {}, {}
if not run.is_done("predict"):
    t0 = time.time()
    for name, df in (("val", val), ("test", test)):
        probs[name] = token_probs(data[name][0])
        feats[name] = pd.concat([df[["id"]], pd.DataFrame([text_features(p) for p in probs[name]])], axis=1)
        feats[name].to_csv(run.pred_path(f"features_{name}"), index=False)
    run.mark_done("predict")
    print(f"{(time.time() - t0) / 60:.1f} min")
else:
    probs["val"] = token_probs(data["val"][0])
feats = {name: pd.read_csv(run.pred_path(f"features_{name}")) for name in ("val", "test")}

In [ ]:
# Token-level quality on validation: AI-class precision/recall/F1 by true label of the text
tok_rows = {}
for lab in LABELS:
    idx = np.flatnonzero((val["label"] == lab).to_numpy())
    tok_rows[lab] = token_scores([probs["val"][i] for i in idx], [data["val"][1][i] for i in idx])
tok_rows["all"] = token_scores(probs["val"], data["val"][1])
run.log_metrics("val_tokens", tok_rows["all"])
print("share of tokens predicted AI:", {lab: round(float(np.mean(np.concatenate(
    [probs["val"][i] for i in np.flatnonzero((val["label"] == lab).to_numpy())]) > 0.5)), 3) for lab in LABELS})
pd.DataFrame(tok_rows).T.round(3)

In [ ]:
# Text features per class on validation (medians): do they separate human, fusion and ai?
feats["val"].drop(columns="id").assign(label=val["label"]).groupby("label").median().T[LABELS].round(3)

In [ ]:
# Distinct warnings logged in this session (should be empty), total time
print(f"total: {(time.time() - NOTEBOOK_START) / 60:.1f} min")
env.warnings_summary(P)